## Multi-Model Translation Comparison (German $\to$ Odia)
This notebook executes a parallel evaluation for the **German $\to$ Odia** translation direction, mirroring the Odia $\to$ German comparison but focusing on the more challenging task of generating low-resource Odia text from German inputs.

### Key Achievements:
* **Head-to-Head Evaluation:** Generated translations for **50 test sentences** using four distinct systems:
  1. **Baseline NLLB:** Pre-trained performance (Zero-shot).
  2. **Full Fine-Tuned (FFT) NLLB:** Specialized on the custom corpus.
  3. **LoRA Fine-Tuned NLLB:** Parameter-efficient adaptation.
  4. **Google Translate:** Commercial reference standard.
* **Consolidated Output:** Produced a unified JSONL file containing the German source and all four Odia translation hypotheses, facilitating direct qualitative comparison of how well each model handles German compound words and Odia sentence structure.

### Workflow Context:
* **Input:** `german_news_clean_final.jsonl` (Subset N=50)
* **Process:** Sequential Inference
* **Output:** `german_to_odia_model_translation_comparison_results.csv`

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# install the library for Google Translate
!pip install -q deep-translator

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 3.9 MB/s eta 0:00:00


In [ ]:
import json
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from peft import PeftModel
from deep_translator import GoogleTranslator
from tqdm import tqdm

In [ ]:
# --- CONFIGURATION ---
CLEANED_FILE = "/content/drive/MyDrive/Research_Paper_Publication/test/data/new_data/cleaned/german_news_clean_final.jsonl"
OUTPUT_FILE = "/content/drive/MyDrive/Research_Paper_Publication/test/eval/german_to_odia_model_translation_comparison_results.csv"

# Paths to your models
FULL_FT_MODEL_PATH = "/content/drive/MyDrive/Research_Paper_Publication/model/nllb-odia-german-translator_model_final_v2"
LORA_ADAPTER_PATH = "/content/drive/MyDrive/Research_Paper_Publication/model/lora-odia-german-translator_v2"
BASE_MODEL_NAME = "facebook/nllb-200-distilled-600M"

# Language Codes
SRC_LANG = "deu_Latn"
TGT_LANG = "ory_Orya"
PREFIX = "translate German to Odia: "

In [ ]:
# Load the data
print("Loading dataset...")
records = []
with open(CLEANED_FILE, 'r', encoding='utf-8') as f:
    records = [json.loads(line) for line in f]

records = records[:50]
print(f"Data slice complete. Processing {len(records)} sentences.")
print(f"First sentence:\n{records[0]}")

Loading dataset...
Data slice complete. Processing 50 sentences.
First sentence:
{'input_text': 'translate German to Odia: Mit rund 42,5 Millionen Euro hat eine Frau aus Baden-Württemberg den höchsten Lotto-Gewinn bei einer Ziehung 6aus49 in Deutschland geholt.', 'target_text': '', 'original_source': 'focus', 'article_context': ''}


In [ ]:
# ---------------------------------------------------------
# 1. GENERATE WITH BASELINE NLLB MODEL (Pre-trained)
# ---------------------------------------------------------
print("\n--- 1/4 Loading Baseline NLLB Model ---")
# Load standard Facebook model
tokenizer_base = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, src_lang=SRC_LANG)
model_base = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL_NAME, device_map="auto")

translator_base = pipeline("translation", model=model_base, tokenizer=tokenizer_base, src_lang=SRC_LANG, tgt_lang=TGT_LANG)

print("Generating Baseline Translations...")
for record in tqdm(records):
    input_text = record.get('input_text')

    # Cleaning: Remove the fine-tuning prefix for the baseline model
    # (Baseline NLLB doesn't know this prefix and might translate it literally)
    clean_input = input_text.replace(PREFIX, "").strip()

    try:
        pred = translator_base(clean_input, max_length=1024, truncation=True)
        record['trans_baseline'] = pred[0]['translation_text']
    except Exception as e:
        record['trans_baseline'] = f"ERROR: {str(e)}"


--- 1/4 Loading Baseline NLLB Model ---


tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Device set to use cuda:0


Generating Baseline Translations...


100%|██████████| 50/50 [00:36<00:00,  1.37it/s]


In [ ]:
# Cleanup
del model_base
del translator_base
del tokenizer_base
torch.cuda.empty_cache()

In [ ]:
# ---------------------------------------------------------
# 2. GENERATE WITH FULL FINE-TUNED MODEL
# ---------------------------------------------------------
print("\n--- 2/4 Loading Full Fine-Tuned Model ---")
tokenizer = AutoTokenizer.from_pretrained(FULL_FT_MODEL_PATH, src_lang=SRC_LANG)
model_fft = AutoModelForSeq2SeqLM.from_pretrained(FULL_FT_MODEL_PATH, device_map="auto")

translator_fft = pipeline("translation", model=model_fft, tokenizer=tokenizer, src_lang=SRC_LANG, tgt_lang=TGT_LANG)

print("Generating Full Fine-Tuned Translations...")
for record in tqdm(records):
    # Truncate to 1024 tokens to avoid errors with long news text
    input_text = record['input_text']
    try:
        # NLLB usually handles raw text well, but you can add your prefix if strictly needed
        pred = translator_fft(input_text, max_length=1024, truncation=True)
        record['trans_full_finetune'] = pred[0]['translation_text']
    except Exception as e:
        record['trans_full_finetune'] = f"ERROR: {str(e)}"


--- 2/4 Loading Full Fine-Tuned Model ---


The tokenizer you are loading from '/content/drive/MyDrive/Research_Paper_Publication/model/nllb-odia-german-translator_model_final_v2' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
Device set to use cuda:0


Generating Full Fine-Tuned Translations...


100%|██████████| 50/50 [00:30<00:00,  1.63it/s]


**Note**:
* This is a false positive warning coming from the Hugging Face transformers library. We are training NLLB (developed by Meta), but the warning explicitly mentions Mistral (a completely different model architecture) and links to a discussion about "Mistral-Small". NLLB uses a SentencePiece tokenizer (Unigram/BPE), which is fundamentally different from the Mistral tokenizer. The "fix" suggested (fix_mistral_regex=True) is irrelevant for NLLB.
* *Why does it happen "now" (during inference) and not during training?* During training we loaded the tokenizer directly from the Hub (facebook/nllb-200-distilled-600M). The original files on the Hub are likely older or formatted in a way that doesn't trigger this specific check.
* During inference we are loading the tokenizer from your local directory (FULL_FT_MODEL_PATH). When Trainer saved our model, it re-serialized the tokenizer into a new tokenizer.json. The structure of this newly saved JSON is what is triggering the "Mistral" safety check in the library.

In [ ]:
# Cleanup to free GPU memory
del model_fft
del translator_fft
torch.cuda.empty_cache()

In [ ]:
# ---------------------------------------------------------
# 3. GENERATE WITH LoRA FINE-TUNED MODEL
# ---------------------------------------------------------
print("\n--- 3/4 Loading LoRA Model (Base + Adapter) ---")
# Load Base Model
model_base = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL_NAME, device_map="auto")
# Load Adapter
model_lora = PeftModel.from_pretrained(model_base, LORA_ADAPTER_PATH)

# Note: Pipeline requires the merged model or explicit handling.
# We use the model directly in the pipeline which works for PEFT in newer versions.
translator_lora = pipeline("translation", model=model_lora, tokenizer=tokenizer, src_lang=SRC_LANG, tgt_lang=TGT_LANG)

print("Generating LoRA Translations...")
for record in tqdm(records):
    input_text = record['input_text']
    try:
        pred = translator_lora(input_text, max_length=1024, truncation=True)
        record['trans_lora'] = pred[0]['translation_text']
    except Exception as e:
        record['trans_lora'] = f"ERROR: {str(e)}"


--- 3/4 Loading LoRA Model (Base + Adapter) ---


Device set to use cuda:0


Generating LoRA Translations...


100%|██████████| 50/50 [00:57<00:00,  1.14s/it]


In [ ]:
# Cleanup
del model_lora
del model_base
del translator_lora
torch.cuda.empty_cache()

In [ ]:
# ---------------------------------------------------------
# 3. GENERATE WITH GOOGLE TRANSLATE API
# ---------------------------------------------------------
print("\n--- 4/4 Generating Google Translations ---")
# Using deep-translator wrapper for ease of use
translator_google = GoogleTranslator(source='de', target='or')

for record in tqdm(records):
    input_text = record['input_text']
    try:
        # Google Translate API has character limits per request (usually ~5000 chars)
        # We split if necessary or just send the text if it's reasonable length
        if len(input_text) > 4500:
             input_text = input_text[:4500] # Simple truncation for safety

        pred = translator_google.translate(input_text)
        record['trans_google'] = pred
    except Exception as e:
        record['trans_google'] = f"ERROR: {str(e)}"


--- 4/4 Generating Google Translations ---


100%|██████████| 50/50 [00:04<00:00, 11.47it/s]


In [ ]:
# ---------------------------------------------------------
# SAVE RESULTS
# ---------------------------------------------------------
print(f"\nSaving all results to {OUTPUT_FILE}...")
with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    for record in records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("Completed Successfully!")


Saving all results to /content/drive/MyDrive/Research_Paper_Publication/test/eval/german_to_odia_model_translation_comparison_results.csv...
Completed Successfully!
